# Cross-Domain Comparison — Finance and Energy

## Trustworthy Time-Series Forecasting Across Domain, Horizon, and Information Regime

## Role

This notebook synthesizes the completed Bitcoin and South Australian Electricity studies
after each domain's forecasts, Regime-Conditional Robustness, Temporal Stability, uncertainty
calibration, trustworthiness, and statistical inference evidence have already been frozen and
validated in their own domain notebooks. It does not repeat those analyses; it reads their
frozen outputs and asks which conclusions replicate across domain, protocol, and horizon, and
which do not.

**Completed tasks**

- **Finance:** Bitcoin daily rolling one-step forecasting.
- **Energy A:** South Australia 30-minute rolling one-step forecasting.
- **Energy B:** South Australia true 48-step / 24-hour day-ahead forecasting.

**Inputs.** The current cross-domain comparison artifacts under `results/` (`cross_domain_*.csv`),
verified against the current Bitcoin and Electricity domain artifacts that back them
(`bitcoin_point_forecast_metrics_v2.csv`, `bitcoin_trustworthiness_components_v2.csv`,
`bitcoin_dm_pairwise_results_hac_holm.csv`, `electricity/protocol_{a,b}_trust_scores.csv`,
`electricity/protocol_{a,b}_dm_tests.csv`, `electricity/protocol_{a,b}_validated_forecasts.csv`,
`foundation_uncertainty_calibration.csv`, `electricity/uncertainty_summary.csv`). Every count,
rank, and score used below is read from these files at run time; none is typed into the notebook
as a literal result.

**Outputs.** This notebook produces cross-task comparison tables, within-task rank comparison,
baseline-relative comparisons, calibration comparison, Regime-Conditional Robustness / Temporal
Stability comparison, trustworthiness profile comparison, statistical-methodology reconciliation,
a small set of cross-domain synthesis figures, and preliminary research conclusions bounded to the
two completed domains.

**Authoritative status.** AUTHORITATIVE CROSS-DOMAIN SYNTHESIS OF THE TWO COMPLETED EMPIRICAL
DOMAINS. The domain-specific notebooks (Bitcoin 09–12; Electricity 15–18) remain authoritative for
their own raw results — this notebook never overrides a domain notebook's own numbers, it only
compares them under an explicit comparability framework.

**What this notebook does not do.** No model fitting. No forecast generation. No raw-unit pooling
of Bitcoin and Electricity errors. No pooled statistical inference across domains. No universal
model ranking that erases task identity. No claim of four-domain generalisation (Weather and
Transport are not evaluated). No causal explanation for why a rank differs between domains — domain
and protocol change simultaneously, so a rank change cannot be attributed to either alone.

# 1. Objective and Research Questions

> Which forecasting conclusions remain stable across Finance and Energy, and which depend
> materially on domain, forecast horizon, or allowable information updates?

**RQ1 — Foundation Model Dominance.** Do Chronos-Bolt-Tiny and TimesFM consistently outperform
strong simple/statistical systems?

**RQ2 — Rank Stability.** How much do model-family ranks change across Bitcoin, Electricity
Protocol A, and Electricity Protocol B?

**RQ3 — Baseline Strength.** How difficult is it for foundation models to beat the strongest
protocol-eligible non-foundation baseline in each task?

**RQ4 — Robustness.** Do aggregate accuracy winners remain strong under regime-conditional stress?

**RQ5 — Temporal Stability.** Do models maintain their relative performance across contiguous
test segments?

**RQ6 — Calibration.** Does better point accuracy correspond to better uncertainty calibration?

**RQ7 — Trustworthiness.** Do component-level trustworthiness dimensions support the same model
preference as the composite score?

**RQ8 — Statistical Evidence.** Are model differences statistically distinguishable within each
task under the domain-specific inference methodology?

**RQ9 — Horizon / Update Discipline.** How strongly do ranking and reliability depend on one-step
updating versus fixed-origin day-ahead forecasting?

# 2. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "results"
RE = R / "electricity"

# Cross-domain synthesis artifacts (already frozen; read-only)
families = pd.read_csv(R / "cross_domain_comparable_families.csv")
not_comparable = pd.read_csv(R / "cross_domain_not_comparable.csv")
comparison = pd.read_csv(R / "cross_domain_model_comparison.csv")
foundation = pd.read_csv(R / "cross_domain_foundation_model_comparison.csv")
trust_cmp = pd.read_csv(R / "cross_domain_trust_comparison.csv")
uncertainty_cmp = pd.read_csv(R / "cross_domain_uncertainty_comparison.csv")
significance_cmp = pd.read_csv(R / "cross_domain_significance_summary.csv")
rank_stability = pd.read_csv(R / "cross_domain_rank_stability.csv")

# Domain-level artifacts needed to derive full-roster ranks and structural counts
# (frozen upstream outputs -- read-only; never recomputed here)
bitcoin_components = pd.read_csv(R / "bitcoin_trustworthiness_components_v2.csv")
bitcoin_dm = pd.read_csv(R / "bitcoin_dm_pairwise_results_hac_holm.csv")
bitcoin_validated = pd.read_csv(R / "validated_forecasts.csv")
bitcoin_uncertainty = pd.read_csv(R / "foundation_uncertainty_calibration.csv")

electricity_trust_a = pd.read_csv(RE / "protocol_a_trust_scores.csv")
electricity_trust_b = pd.read_csv(RE / "protocol_b_trust_scores.csv")
electricity_dm_a = pd.read_csv(RE / "protocol_a_dm_tests.csv")
electricity_dm_b = pd.read_csv(RE / "protocol_b_dm_tests.csv")
electricity_validated_a = pd.read_csv(RE / "protocol_a_validated_forecasts.csv")
electricity_validated_b = pd.read_csv(RE / "protocol_b_validated_forecasts.csv")
electricity_uncertainty = pd.read_csv(RE / "uncertainty_summary.csv")

TASKS = ["Bitcoin", "Electricity A", "Electricity B"]

In [ ]:
# Structural counts derived from artifacts -- never typed as literals
n_bitcoin_models = int((comparison.Domain == "Bitcoin").sum())
n_electricity_a_models = int(((comparison.Domain == "Electricity") & comparison.Protocol.str.contains("A")).sum())
n_electricity_b_models = int(((comparison.Domain == "Electricity") & comparison.Protocol.str.contains("B")).sum())
ROSTER_SIZE = {"Bitcoin": n_bitcoin_models, "Electricity A": n_electricity_a_models, "Electricity B": n_electricity_b_models}

n_bitcoin_test = int(len(bitcoin_validated))
n_electricity_a_test = int(len(electricity_validated_a))
n_electricity_b_rows = int(len(electricity_validated_b))
n_electricity_b_origins = int(electricity_validated_b["Origin"].nunique())

def normalized_rank(rank, n):
    # 0 = best, 1 = worst; descriptive transformation only, not a new quality metric
    return (rank - 1) / (n - 1) if n > 1 else 0.0

display(pd.DataFrame([{
    "Task": t, "Model_Count": ROSTER_SIZE[t]
} for t in TASKS]))